# Entrega 1: Proyecto Soluciones Intensivas en Datos (2026 -02)

%md
# Notebook 2: Lectura, Exploración y Perfilamiento de los datos

Previo a la ejecución de este notebook, se realizó la ingesta de la información con el notebook **Notebook1_IngestaDatos**

## OceanWatch Analytics

### Presentado por: Gupo 2

#### Pablo Daniel Díaz Mera
#### Ana Catalina Gelvez
#### Christian Rene Mendez

## 1. Lectura de los datos

Teniendo en cuenta el enunciado, se realiza la lectura de los datos descargados en el notebook 1 teniendo en cuenta los atributos solicitados. Sin embargo, primero se realiza la verificación del encabezado de uno de los csv para validar la nomenclatura exacta

In [0]:
RUTA_CSV = "/Volumes/workspace/oceanwatch/ais_data/csv"

archivo_muestra = f"{RUTA_CSV}/AIS_2023_06_01.csv"

with open(archivo_muestra, "r") as archivo:
    encabezado = archivo.readline().strip()

print(encabezado)

MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,VesselType,Status,Length,Width,Draft,Cargo,TransceiverClass


- Una vez verificados los encabezados, se define explícitamente el esquema que se utilizará para la lectura.
- De esta manera, se evita que spark realice la inferencia lo cual es computacionalmente costoso.

Definición explícita del esquema

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    TimestampType
)
#Definición del esquema
esquema_ais = StructType([
    StructField("MMSI", IntegerType(), True),
    StructField("BaseDateTime", TimestampType(), True),
    StructField("LAT", DoubleType(), True),
    StructField("LON", DoubleType(), True),
    StructField("SOG", DoubleType(), True),
    StructField("COG", DoubleType(), True),
    StructField("Heading", DoubleType(), True),
    StructField("VesselName", StringType(), True),
    StructField("IMO", StringType(), True),
    StructField("CallSign", StringType(), True),
    StructField("VesselType", IntegerType(), True),
    StructField("Status", IntegerType(), True),
    StructField("Length", DoubleType(), True),
    StructField("Width", DoubleType(), True),
    StructField("Draft", DoubleType(), True),
    StructField("Cargo", IntegerType(), True),
    StructField("TransceiverClass", StringType(), True)
])

In [0]:
ais = (
    spark.read
        .option("header", "true")
        .schema(esquema_ais)
        .csv(f"{RUTA_CSV}/*.csv")
)

#Verificación del esquema
ais.printSchema()

root
 |-- MMSI: integer (nullable = true)
 |-- BaseDateTime: timestamp (nullable = true)
 |-- LAT: double (nullable = true)
 |-- LON: double (nullable = true)
 |-- SOG: double (nullable = true)
 |-- COG: double (nullable = true)
 |-- Heading: double (nullable = true)
 |-- VesselName: string (nullable = true)
 |-- IMO: string (nullable = true)
 |-- CallSign: string (nullable = true)
 |-- VesselType: integer (nullable = true)
 |-- Status: integer (nullable = true)
 |-- Length: double (nullable = true)
 |-- Width: double (nullable = true)
 |-- Draft: double (nullable = true)
 |-- Cargo: integer (nullable = true)
 |-- TransceiverClass: string (nullable = true)



se procede a hacer una visualización inicial de los datos

In [0]:
display(ais.limit(10))

MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,VesselType,Status,Length,Width,Draft,Cargo,TransceiverClass
367138690,2023-06-01T00:00:01.000Z,30.28335,-91.22283,0.0,273.0,511.0,ST FRANCISVILLE,null,WDD3968,60,0,43.0,18.0,null,60,A
368036960,2023-06-01T00:00:01.000Z,40.49818,-80.07596,3.3,118.6,131.0,GALE R RHODES,IMO8199788,WDK2590,0,12,20.0,7.0,2.6,0,A
368095340,2023-06-01T00:00:01.000Z,29.99678,-93.94959,0.0,309.8,511.0,GAMBLER,null,WDK8562,52,15,19.0,8.0,0.0,52,A
248301000,2023-06-01T00:00:03.000Z,26.13342,-80.11909,0.0,260.5,86.0,ATLAS,null,9HB5599,37,5,31.0,8.0,3.0,37,A
367186410,2023-06-01T00:00:03.000Z,30.00762,-93.95571,5.7,164.0,164.0,BRIAN O DANIELS,null,WDD7438,31,12,26.0,9.0,null,31,A
367108820,2023-06-01T00:00:05.000Z,37.82833,-76.27936,0.1,235.2,511.0,TIDELANDS,IMO7309443,WX5026,30,0,61.0,9.0,null,30,A
367320740,2023-06-01T00:00:05.000Z,38.73458,-82.87265,4.7,350.3,349.0,TENNESSEE,IMO8836584,WDE2124,31,0,42.0,12.0,null,52,A
368101220,2023-06-01T00:00:03.000Z,32.6922,-117.14808,0.0,360.0,511.0,WILLIAM F,null,WDK9183,33,15,40.0,18.0,0.0,33,A
367529860,2023-06-01T00:00:08.000Z,29.61437,-95.00393,0.2,197.6,285.0,C R MCCASKILL,null,WDG3808,90,3,67.0,null,null,33,A
367558580,2023-06-01T00:00:09.000Z,27.3326,-82.54605,0.0,233.2,221.0,DAYS LIKE THIS,null,WDG6586,37,0,40.0,7.0,null,37,A


- Se identifica que los datos se leyeron de manera correcta los tipos de datos coinciden con los definidos en el esquema
- Se encuentra que existen valores nulos.

## 2. Exploración y perfilamiento de los datos

### Posiciones registradas por día

- Inicialmente, se realiza la selección sólamente de la columna que nos interesa (esto evita leer todas las columnas)
- Posteriormente, se obtiene la fecha a partir del BaseDateTime y al final se agrupan los registros por día.

In [0]:
from pyspark.sql import functions as f

posiciones_dia = (
    ais
    #Se define que solo se va a tomar la columna BaseDateTime
    .select("BaseDateTime")
    .groupBy(
        f.to_date("BaseDateTime").alias("fecha")
    )
    .agg(
        f.count("*").alias("posiciones")
    )
)

display(posiciones_dia.orderBy("fecha"))

fecha,posiciones
2023-06-01,8808904
2023-06-02,9052241
2023-06-03,8036348
2023-06-04,8522645
2023-06-05,8613757
2023-06-06,8587938
2023-06-07,8911726


Se encuentra que se leyeron 60.533559 filas y 6.04 GB
La ejecución duró 21 s 88 ms

- Se encuentra que diariamente se registran entre 8 y 9,1 millones de posiciones.
- El día 2 de junio fue el día con una mayor cantidad de posiciones (9.052.241).
- El día 3 de junio fue el día con una menor cantidad de posiciones (8.036.348)

### Buques únicos por día y en toda la semana

- Para identificar este valor, se utiliza el atributo MMSI (identificador del buque)
- Al tratarse de un perfilamiento inical de los datos, se realiza un conteo aproximado para evitar el costo computacional de tener los datos exactos de un conteo de valores distintos.
- Al igual que en el conteo de posiciones por día, se utilizan solo las columnas de interés.

#### Buques únicos por día

In [0]:
buques_dia = (
    ais
    .select("BaseDateTime", "MMSI")
    .groupBy(
        f.to_date("BaseDateTime").alias("fecha")
    )
    .agg(
        f.approx_count_distinct("MMSI", 0.01)
        .alias("buques_unicos_aprox")
    )
)

display(buques_dia.orderBy("fecha"))

fecha,buques_unicos_aprox
2023-06-01,20445
2023-06-02,21274
2023-06-03,20425
2023-06-04,19780
2023-06-05,19380
2023-06-06,19650
2023-06-07,19992


- El tiempo de ejecución fue de 20 s 710 ms, se leyeron 60.533.559 filas y se leyeron 6.04 GB
6.04 GB
- Se encuentra que el día 2 de junio fue el día con un mayor número de buques únicos

#### Buques únicos por semana

In [0]:
buques_semana = (
    ais
    .select("MMSI")
    .agg(
        f.approx_count_distinct("MMSI", 0.01)
        .alias("buques_unicos_semana_aprox")
    )
)

display(buques_semana)

buques_unicos_semana_aprox
31691



El tiempo de ejecución fue de 18 s 881 ms, se leyeron 60.533.559 filas y 6.04 GB

La diferencia entre los conteos diarios y el total semanal indica que existe rotación de embarcaciones durante el periodo, ya que no todos los buques transmiten información durante los siete días.

### Distribución por tipo de buque

- Para este caso, se utilizan las columnas MMSI y VesselType
- Se estima la cantidad de buques únicos (MMSI) por cada tipo (VesselType)
- Igual que en el punto anterior, se realiza el cálculo aproximado con un rsd = 0,01.

In [0]:
buques_tipo = (
    ais
    .select("MMSI", "VesselType")
    .groupBy("VesselType")
    .agg(
        f.approx_count_distinct("MMSI", 0.01)
        .alias("buques_unicos_aprox")
    )
)

display(
    buques_tipo.orderBy(
        f.desc("buques_unicos_aprox")
    )
)

VesselType,buques_unicos_aprox
37,12952
36,4245
31,3477
30,2197
60,1702
70,1568
90,1530
null,855
80,822
52,481


- Se encuentra que el tipo de buque de mayor cantidad corresponde al Código 37, que corresponde a Pleasure Craft
 (https://coast.noaa.gov/data/marinecadastre/ais/VesselTypeCodes2018.pdf). que son embarcaciones de recreo o de placer.
- De igual manera, se encuentra que hay 885 embarcaciones que con VesselType nulos.


### Distribución por tamaño

- Para este caso, se va a tomar la longitud (columna Length) como referencia para hacer la clasificación.
- Se realizará el calculo de estadísticas descriptivas para conocer la distribución de la variable.

#### Estadísticas descriptivas de la longitud de los buques

In [0]:
perfil_tamano = (
    ais
    .select("Length")
    .agg(
        f.min("Length").alias("longitud_min"),
        f.expr("percentile_approx(Length, 0.25)").alias("p25"),
        f.expr("percentile_approx(Length, 0.50)").alias("mediana"),
        f.expr("percentile_approx(Length, 0.75)").alias("p75"),
        f.avg("Length").alias("longitud_promedio"),
        f.max("Length").alias("longitud_max")
    )
)

display(perfil_tamano)

longitud_min,p25,mediana,p75,longitud_promedio,longitud_max
0.0,15.0,22.0,38.0,48.62184200928051,638.0


- La mediana es de 22 metros de longitud
- El 75% de los registros tiene una longitud de 38 metros o menos
- La longitud promedio de los buques es de 48,62 metros
- Al revisar los valores mínimo (0 metros) y máximo (638 metros), se encuentra que existen valores que deberían revisarse porque podrían ser datos anomalos.


### Problemas de calidad

#### Conteo de valores nulos

En este caso, se tienen en cuenta todas las columnas debido a que se desea hacer una validación de todo el conjunto de datos

In [0]:
conteo_nulos = ais.agg(
    *[
        f.sum(
            f.when(f.col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in ais.columns
    ]
)

display(conteo_nulos)

MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,VesselType,Status,Length,Width,Draft,Cargo,TransceiverClass
0,0,0,0,0,0,0,150685,25891448,10157915,209863,19952397,3664725,9219567,39000865,19898489,0



tiempo de ejecución 29 s 193 ms, filas leídas 60.533.559 y 6.04 GB leídos

- Se encuentra que los atributos importantes para identificar y localizar los buques se encuentran completos, (MMSI, BaseDateTime, LAT, LON).
- Varios de los atributos descriptivos tienen una cantidad considerable de nulos.

#### Porcentaje de valores nulos

In [0]:
TOTAL_REGISTROS = 60533559

porcentaje_nulos = conteo_nulos.select(
    *[
        f.round(
            (f.col(c) / f.lit(TOTAL_REGISTROS)) * 100,
            2
        ).alias(c)
        for c in conteo_nulos.columns
    ]
)

display(porcentaje_nulos)

MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,VesselType,Status,Length,Width,Draft,Cargo,TransceiverClass
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,42.77,16.78,0.35,32.96,6.05,15.23,64.43,32.87,0.0


- Se encuentra que el campo Draft, tiene un 64,43 % de valores nulos mientras que campos como Cargo, Status e IMO tienen un porcentaje de valores nulos superior al 30 %.

#### Posiciones geográficas por fuera de rango

Inicialmente, se valida la distribución de LAT y LONG.

In [0]:
perfil_coordenadas = (
    ais
    .select("LAT", "LON")
    .agg(
        f.min("LAT").alias("lat_min"),
        f.expr("percentile_approx(LAT, 0.25)").alias("lat_p25"),
        f.expr("percentile_approx(LAT, 0.50)").alias("lat_mediana"),
        f.expr("percentile_approx(LAT, 0.75)").alias("lat_p75"),
        f.avg("LAT").alias("lat_promedio"),
        f.max("LAT").alias("lat_max"),

        f.min("LON").alias("lon_min"),
        f.expr("percentile_approx(LON, 0.25)").alias("lon_p25"),
        f.expr("percentile_approx(LON, 0.50)").alias("lon_mediana"),
        f.expr("percentile_approx(LON, 0.75)").alias("lon_p75"),
        f.avg("LON").alias("lon_promedio"),
        f.max("LON").alias("lon_max")
    )
)

display(perfil_coordenadas)

lat_min,lat_p25,lat_mediana,lat_p75,lat_promedio,lat_max,lon_min,lon_p25,lon_mediana,lon_p75,lon_promedio,lon_max
0.62368,29.70649,33.73437,41.19049,35.43395040750764,89.6552,-179.95459,-117.13342,-90.14042,-79.45712,-93.40829838103605,146.3172


Se encuentra que las coordenadas observadas,se encuentran dentro de los rangos válidos:
- La latitud varía entre 0,62368 y 89,6552 grados
- la longitud se encuentra entre -179,95459 y 146,3172 grados

In [0]:
posiciones_fuera_rango = (
    ais
    .select("LAT", "LON")
    .agg(
        f.sum(
            f.when(
                (f.col("LAT") < -90) | (f.col("LAT") > 90),
                1
            ).otherwise(0)
        ).alias("latitud_fuera_rango"),

        f.sum(
            f.when(
                (f.col("LON") < -180) | (f.col("LON") > 180),
                1
            ).otherwise(0)
        ).alias("longitud_fuera_rango"),

        f.sum(
            f.when(
                (f.col("LAT") < -90) |
                (f.col("LAT") > 90) |
                (f.col("LON") < -180) |
                (f.col("LON") > 180),
                1
            ).otherwise(0)
        ).alias("posiciones_fuera_rango")
    )
)

display(posiciones_fuera_rango)

latitud_fuera_rango,longitud_fuera_rango,posiciones_fuera_rango
0,0,0


Es importante tener en cuenta que los rangos evaluados corresponden a los límites geográficos globales de latitud (-90 a 90) y longitud (-180 a 180). Esta validación permite identificar coordenadas imposibles a nivel mundial, pero no determina si las posiciones se encuentran específicamente dentro del área marítima de Estados Unidos.

#### Velocidades imposibles

- Inicialmente, se realiza el cálculo de estadísticas descriptivas del campo SOG (velocidad sobre el fondo, nudos)

In [0]:
perfil_velocidad = (
    ais
    .select("SOG")
    .agg(
        f.min("SOG").alias("sog_min"),
        f.expr("percentile_approx(SOG, 0.25)").alias("sog_p25"),
        f.expr("percentile_approx(SOG, 0.50)").alias("sog_mediana"),
        f.expr("percentile_approx(SOG, 0.75)").alias("sog_p75"),
        f.expr("percentile_approx(SOG, 0.95)").alias("sog_p95"),
        f.expr("percentile_approx(SOG, 0.99)").alias("sog_p99"),
        f.avg("SOG").alias("sog_promedio"),
        f.max("SOG").alias("sog_max")
    )
)

display(perfil_velocidad)

sog_min,sog_p25,sog_mediana,sog_p75,sog_p95,sog_p99,sog_promedio,sog_max
0.0,0.0,0.0,3.1,12.5,22.0,2.6289088305546375,102.3


- La velocidad presenta una distribución concentrada en valores bajos. La mediana es 0 y el 75 % de las posiciones registra velocidades de hasta 3,1, mientras que los percentiles 95 y 99 alcanzan 12,5 y 22 respectivamente.
- Por otro lado, Se observa un valor máximo de 102,3 el cual podría considerarse como imposible. Sin embargo, inicialmente se realizará la cuantificación de registros con este valor

In [0]:
velocidades_altas = (
    ais
    .select("SOG")
    .agg(
        f.sum(f.when(f.col("SOG") > 22, 1).otherwise(0))
            .alias("mayor_22"),

        f.sum(f.when(f.col("SOG") > 30, 1).otherwise(0))
            .alias("mayor_30"),

        f.sum(f.when(f.col("SOG") > 50, 1).otherwise(0))
            .alias("mayor_50"),

        f.sum(f.when(f.col("SOG") > 100, 1).otherwise(0))
            .alias("mayor_100"),

        f.sum(f.when(f.col("SOG") == 102.3, 1).otherwise(0))
            .alias("igual_102_3")
    )
)

display(velocidades_altas)

mayor_22,mayor_30,mayor_50,mayor_100,igual_102_3
601794,265238,161418,160280,159987


- Se identificaron 601.794 registros con velocidades superiores a 22 nudos, 265.238 superiores a 30 y 161.418 superiores a 50
- La mayor parte de los valores extremos se concentra alrededor de 102 nudos
- Al tener una cantidad tan alta de registros con valores altos, no se realiza una eliminación. Por ahora, se conservan y se propone realizar una investigación mas a fondo de las características del dataset en documentación asociada.


#### Identificadores anómalos

- Se inicia con el cálculo de estadísticas descriptivas para el campo MMSI
- Se realiza un conteo aproximado de valores únicos

In [0]:
perfil_mmsi = (
    ais
    .select("MMSI")
    .agg(
        f.min("MMSI").alias("mmsi_min"),
        f.max("MMSI").alias("mmsi_max"),
        f.approx_count_distinct("MMSI", 0.01)
            .alias("mmsi_unicos_aprox")
    )
)

display(perfil_mmsi)

mmsi_min,mmsi_max,mmsi_unicos_aprox
0,1072339762,31691



- Teniendo en cuenta la descripción del código MMSI:

_"The nine digit unique ship station identity is in the format MIDXXXXXX where in the first three digits represent the Maritime Identification Digits (MID) and X is any figure from 0 to 9, as indicated in Recommendation ITU-R M.585. The MID denotes the administration having jurisdiction over the ship station so identified."_ (https://www.itu.int/en/ITU-R/terrestrial/Pages/by-categories-faq.aspx?categorizedby=17)

- Se identifica que los códigos deberían tener una longitud de 9 dígitos pero se encuentra que el valor mínimo del dataset (o) y el valor máximo (1072339762), no cuentan con la estructura definida.
- Se procede a hacer la cuantificación de identifiadores con mas o menos de nueve dígitos

In [0]:
mmsi_anomalos = (
    ais
    .select("MMSI")
    .agg(
        f.sum(
            f.when(f.col("MMSI") == 0, 1).otherwise(0)
        ).alias("mmsi_cero"),

        f.sum(
            f.when(
                (f.col("MMSI") > 0) &
                (f.col("MMSI") < 100000000),
                1
            ).otherwise(0)
        ).alias("mmsi_menos_9_digitos"),

        f.sum(
            f.when(
                f.col("MMSI") > 999999999,
                1
            ).otherwise(0)
        ).alias("mmsi_mas_9_digitos"),

        f.sum(
            f.when(
                (f.col("MMSI") < 100000000) |
                (f.col("MMSI") > 999999999),
                1
            ).otherwise(0)
        ).alias("mmsi_longitud_anomala")
    )
)

display(mmsi_anomalos)

mmsi_cero,mmsi_menos_9_digitos,mmsi_mas_9_digitos,mmsi_longitud_anomala
978,45794,3125,49897


- Se identifican 49.897 registros cuyo MMSI no cumple con la estructura esperada.
- Se debe recordar que este conteo no corresponde a MMSI únicos.
- Se procede a contar los MMSI únicos que no cumplen con la estructura. Para esto, primero hacemos el filtro y luego el groupby.

In [0]:
mmsi_anomalos_detalle = (
    ais
    .select("MMSI")
    .filter(
        (f.col("MMSI") < 100000000) |
        (f.col("MMSI") > 999999999)
    )
    .groupBy("MMSI")
    .agg(
        f.count("*").alias("registros")
    )
)

mmsi_anomalos_unicos = mmsi_anomalos_detalle.count()

print(f"MMSI anómalos únicos: {mmsi_anomalos_unicos}")

MMSI anómalos únicos: 91


- Los 49.897 registros con una longitud anómala de MMSI corresponden a 91 identificadores únicos. Esto indica que el problema se concentra en un número reducido de identificadores

#### Identidicación de duplicados

In [0]:
duplicados_exactos = (
    ais
    .groupBy(*ais.columns)
    .agg(
        f.count("*").alias("repeticiones")
    )
    .filter(f.col("repeticiones") > 1)
)

In [0]:
resumen_duplicados = (
    duplicados_exactos
    .agg(
        f.count("*").alias("grupos_duplicados"),
        f.sum("repeticiones").alias("registros_en_grupos_duplicados"),
        f.sum(f.col("repeticiones") - 1).alias("duplicados_excedentes")
    )
)

display(resumen_duplicados)

grupos_duplicados,registros_en_grupos_duplicados,duplicados_excedentes
1388,2776,1388


- Se identificaron 1.388 grupos de registros completamente duplicados, que corresponden a 2.776 filas involucradas.
- Se procede a verificar duplicados por la combinación MMSI + BaseDateTime. Dos filas podrían no ser idénticas en las 17 columnas, pero corresponder al mismo buque en el mismo instante.

In [0]:
duplicados_mmsi_tiempo = (
    ais
    .select("MMSI", "BaseDateTime")
    .groupBy("MMSI", "BaseDateTime")
    .agg(
        f.count("*").alias("repeticiones")
    )
    .filter(f.col("repeticiones") > 1)
)

In [0]:
resumen_duplicados_mmsi_tiempo = (
    duplicados_mmsi_tiempo
    .agg(
        f.count("*").alias("grupos_duplicados"),
        f.sum("repeticiones").alias("registros_involucrados"),
        f.sum(f.col("repeticiones") - 1).alias("registros_excedentes"),
        f.max("repeticiones").alias("max_repeticiones")
    )
)

display(resumen_duplicados_mmsi_tiempo)

grupos_duplicados,registros_involucrados,registros_excedentes,max_repeticiones
1672,3344,1672,2


- Se identificaron 1.672 combinaciones duplicadas de MMSI y BaseDateTime
- Este número es superior a los 1.388 duplicados exactos encontrados previamente, lo que indica la existencia de registros que coinciden en buque y momento de transmisión.

## 3. Preguntas de negocio

### a. ¿Cuántos buques distintos transmitieron cada día?

Comparen el conteo exacto con approx_count_distinct y argumenten cuál usarían en
producción.

Se calcula la cantidad de buques distintos que transmitieron posiciones AIS cada día utilizando dos métodos: countDistinct, que obtiene la cardinalidad exacta de MMSI, y approx_count_distinct

Conteo exacto

In [0]:
buques_dia_exacto = (
    ais
    .select("BaseDateTime", "MMSI")
    .groupBy(
        f.to_date("BaseDateTime").alias("fecha")
    )
    .agg(
        f.countDistinct("MMSI").alias("buques_unicos")
    )
)

display(buques_dia_exacto.orderBy("fecha"))

fecha,buques_unicos
2023-06-01,20448
2023-06-02,21153
2023-06-03,20505
2023-06-04,19720
2023-06-05,19615
2023-06-06,19717
2023-06-07,20086


In [0]:
buques_dia_exacto.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (16)
+- == Initial Plan ==
   PhotonResultStage (15)
   +- PhotonColumnarToRow (14)
      +- PhotonGroupingAgg (13)
         +- PhotonShuffleExchangeSource (12)
            +- PhotonShuffleMapStage (11)
               +- PhotonShuffleExchangeSink (10)
                  +- PhotonGroupingAgg (9)
                     +- PhotonGroupingAgg (8)
                        +- PhotonShuffleExchangeSource (7)
                           +- PhotonShuffleMapStage (6)
                              +- PhotonShuffleExchangeSink (5)
                                 +- PhotonGroupingAgg (4)
                                    +- PhotonProject (3)
                                       +- PhotonRowToColumnar (2)
                                          +- Scan csv  (1)


(1) Scan csv 
Output [2]: [MMSI#187548, BaseDateTime#187549]
Batched: false
Location: InMemoryFileIndex [dbfs:/Volumes/workspace/oceanwatch/ais_data/csv/AIS_2023_06_01.csv, ... 6 entries]
ReadSchema: s

Conteo aproximado

In [0]:
buques_dia_aprox = (
    ais
    .select("BaseDateTime", "MMSI")
    .groupBy(
        f.to_date("BaseDateTime").alias("fecha")
    )
    .agg(
        f.approx_count_distinct("MMSI", 0.01)
        .alias("buques_unicos_aprox")
    )
)

display(
    buques_dia_aprox.orderBy("fecha")
)

fecha,buques_unicos_aprox
2023-06-01,20445
2023-06-02,21274
2023-06-03,20425
2023-06-04,19780
2023-06-05,19380
2023-06-06,19650
2023-06-07,19992


In [0]:
buques_dia_aprox.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (11)
+- == Initial Plan ==
   PhotonResultStage (10)
   +- PhotonColumnarToRow (9)
      +- PhotonGroupingAgg (8)
         +- PhotonShuffleExchangeSource (7)
            +- PhotonShuffleMapStage (6)
               +- PhotonShuffleExchangeSink (5)
                  +- PhotonGroupingAgg (4)
                     +- PhotonProject (3)
                        +- PhotonRowToColumnar (2)
                           +- Scan csv  (1)


(1) Scan csv 
Output [2]: [MMSI#187548, BaseDateTime#187549]
Batched: false
Location: InMemoryFileIndex [dbfs:/Volumes/workspace/oceanwatch/ais_data/csv/AIS_2023_06_01.csv, ... 6 entries]
ReadSchema: struct<MMSI:int,BaseDateTime:timestamp>

(2) PhotonRowToColumnar
Input [2]: [MMSI#187548, BaseDateTime#187549]

(3) PhotonProject
Input [2]: [MMSI#187548, BaseDateTime#187549]
Arguments: [MMSI#187548, cast(BaseDateTime#187549 as date) AS _groupingexpression#329660]

(4) PhotonGroupingAgg
Input [2]: [MMSI#187548, _groupingexpression

**Conteo exacto:**
- Se ejecutan 9 tareas, con un tiempo de ejecución de 22 s 781 ms, se leen 60.533.559 filas con un peso de 6.04 GB

**Conteo aproximado:**
- Se ejecutan 8 tareas, con un tiempo de ejecución de 18 s 44 ms, se leen 60.533.559 filas con un peso de 6.04 GB

- Se encuentra que los resultados obtenidos por los dos métodos son bastante parecidos.
- El tiempo de ejecución del conteo aproximado redujo el tiempo de ejecución aproximadamente un 20,8 % y simplificó el plan de ejecución.
- El conteo exacto se debería usar en casos en los que la exactitud del número de buques es un requsito de negocio no en todos los casos.

### b. ¿Qué tipos de buque generan más tráfico?
Top 10 por número de posiciones, con velocidad media por tipo (usen el catálogo de tipos).

- Se va a calcular el tráfico generado por cada tipo de buque utilizando el número de posiciones AIS registradas.
- Para cada código VesselType también se obtiene la velocidad media (SOG)
- FInalmente, los códigos del top10 se asociarán con el catálogo de tipos.

In [0]:
trafico_tipo = (
    ais
    .select("VesselType", "SOG")
    .filter(f.col("VesselType").isNotNull())
    .groupBy("VesselType")
    .agg(
        f.count("*").alias("numero_posiciones"),
        f.avg("SOG").alias("velocidad_media")
    )
)

top10_trafico_tipo = (
    trafico_tipo
    .orderBy(f.desc("numero_posiciones"))
    .limit(10)
)

display(top10_trafico_tipo)

VesselType,numero_posiciones,velocidad_media
31,16558120,1.8987526240930621
37,14476696,1.6422352103023863
60,4795209,4.433895873989091
30,4010188,2.466139193474545
36,3876264,1.9792365019505584
90,3848162,2.0281950188159845
70,3837926,6.369827792410587
52,1921565,2.093070023651827
80,1789168,5.874565719933229
57,1323765,1.9109420478708936


Creación del dataframe con los códigos y descripción del tipo de embarcación de los 10 primeros

In [0]:
catalogo_tipos = spark.createDataFrame(
    [
        (30, "Fishing"),
        (31, "Towing"),
        (36, "Sailing"),
        (37, "Pleasure craft"),
        (52, "Tug"),
        (57, "Spare / local vessel assignment"),
        (60, "Passenger"),
        (70, "Cargo"),
        (80, "Tanker"),
        (90, "Other type")
    ],
    ["VesselType", "tipo_buque"]
)

Join del tipo de embarcación sobre los datos agregados

In [0]:
top10_trafico_catalogo = (
    top10_trafico_tipo
    .join(
        f.broadcast(catalogo_tipos),
        on="VesselType",
        how="left"
    )
    .select(
        "VesselType",
        "tipo_buque",
        "numero_posiciones",
        f.round("velocidad_media", 2).alias("velocidad_media")
    )
    .orderBy(f.desc("numero_posiciones"))
)

display(top10_trafico_catalogo)

VesselType,tipo_buque,numero_posiciones,velocidad_media
31,Towing,16558120,1.9
37,Pleasure craft,14476696,1.64
60,Passenger,4795209,4.43
30,Fishing,4010188,2.47
36,Sailing,3876264,1.98
90,Other type,3848162,2.03
70,Cargo,3837926,6.37
52,Tug,1921565,2.09
80,Tanker,1789168,5.87
57,Spare / local vessel assignment,1323765,1.91


- El tipo de buque que genera mayor tráfico AIS es Towing, con 16.558.120 posiciones, seguido por Pleasure craft, con 14.476.696.
- Respecto a la velocidad media, los buques Cargo presentan el mayor valor entre los diez tipos analizados, con 6,37 nudos, seguidos por Tanker, con 5,87 nudos.

### c. ¿Qué 10 buques recorrieron más distancia durante la semana?
(orden por buque y tiempo, distancia entre posiciones consecutivas con haversine).

- Para estimar la distancia recorrida por cada buque, se van a organizar de manera cronológica las posiciones por cada MMSI
- Se utiliza una ventana haciendo partición por buque para tener las coordenadas de la posición anterior.

In [0]:
from pyspark.sql.window import Window

#definición de la ventana particionada por MMSI
ventana_buque = (
    Window
    .partitionBy("MMSI")
    .orderBy("BaseDateTime")
)

posiciones_consecutivas = (
    ais
    .select(
        "MMSI",
        "BaseDateTime",
        "LAT",
        "LON"
    )
    .withColumn(
        "lat_anterior",
        f.lag("LAT").over(ventana_buque)
    )
    .withColumn(
        "lon_anterior",
        f.lag("LON").over(ventana_buque)
    )
    .filter(
        f.col("lat_anterior").isNotNull() &
        f.col("lon_anterior").isNotNull()
    )
)

- Con esta ventana definida, se calcula la distancia recorrida entre cada par de coordenadas.
- El cálculo se realiza utilizando la fórmula Haversine. Para esto, se utilizará funciones nativas de Spark para no utilizar UDFs.
- Se toma como radio de la tierra el valor de 6.371 km (https://en-wikipedia-org.translate.goog/wiki/Earth_radius?_x_tr_sl=en&_x_tr_tl=es&_x_tr_hl=es&_x_tr_pto=tc)

In [0]:
#definición de la variable del radio de la tierra para la fórmula
RADIO_TIERRA_KM = 6371.0

#Cálculo de las distancias     
distancias = (
    posiciones_consecutivas
    .withColumn(
        "dlat",
        f.radians(f.col("LAT") - f.col("lat_anterior"))
    )
    .withColumn(
        "dlon",
        f.radians(f.col("LON") - f.col("lon_anterior"))
    )
    .withColumn(
        "a",
        f.pow(f.sin(f.col("dlat") / 2), 2)
        +
        f.cos(f.radians(f.col("lat_anterior")))
        * f.cos(f.radians(f.col("LAT")))
        * f.pow(f.sin(f.col("dlon") / 2), 2)
    )
    .withColumn(
        "distancia_km",
        2 * f.lit(RADIO_TIERRA_KM)
        * f.asin(
            f.sqrt(
                f.least(
                    f.lit(1.0),
                    f.col("a")
                )
            )
        )
    )
)

- Las distancias entre posiciones consecutivas se suman para cada MMSI. De esta manera, se estima la distancia total recorrida.
- El resultado se ordena después de la agregación para seleccionar los diez buques con mayor distancia recorrida

In [0]:
distancia_por_buque = (
    distancias
    .groupBy("MMSI")
    .agg(
        f.sum("distancia_km").alias("distancia_total_km")
    )
)

top10_distancia = (
    distancia_por_buque
    .orderBy(f.desc("distancia_total_km"))
    .limit(10)
)

display(
    top10_distancia
    .withColumn(
        "distancia_total_km",
        f.round("distancia_total_km", 2)
    )
)

MMSI,distancia_total_km
211002010,7931112.22
368124160,7361374.95
367373000,5724082.31
338333275,5065029.78
374158000,3027501.81
338561000,1613419.34
360000000,1155988.08
338479554,852140.41
367530470,317418.18
338182085,249765.18


- Se encuentra que los valores reportados de la distancia recorrida por buque no son valores racionales. Por ejemplo, para el top 1, un desplazamiento de una semana de 7,9 millones de kilometros no hace mucho sentido. Esto puede suceder porque las coordenadas reportadas pueden presentar inconsistencias de señal del gps de cada buque y reportar saltos de distancia de kilómetros en segundos.
- Para diagnosticar esos casos, se calcula el tiempo entre cada par de posiciones y la velocidad necesaria para recorrer la distancia estimada entre cada par de posiciones. Con esto, se pueden diferenciar desplazamientos posibles o imposibles.

In [0]:
posiciones_consecutivas = (
    ais
    .select(
        "MMSI",
        "BaseDateTime",
        "LAT",
        "LON"
    )
    .withColumn(
        "tiempo_anterior",
        f.lag("BaseDateTime").over(ventana_buque)
    )
    .withColumn(
        "lat_anterior",
        f.lag("LAT").over(ventana_buque)
    )
    .withColumn(
        "lon_anterior",
        f.lag("LON").over(ventana_buque)
    )
    .filter(
        f.col("tiempo_anterior").isNotNull()
    )
)

- Se realiza el cálculo de distancias y se agrega la diferencia temporal

In [0]:
distancias = (
    posiciones_consecutivas
    .withColumn(
        "dlat",
        f.radians(f.col("LAT") - f.col("lat_anterior"))
    )
    .withColumn(
        "dlon",
        f.radians(f.col("LON") - f.col("lon_anterior"))
    )
    .withColumn(
        "a",
        f.pow(f.sin(f.col("dlat") / 2), 2)
        +
        f.cos(f.radians("lat_anterior"))
        * f.cos(f.radians("LAT"))
        * f.pow(f.sin(f.col("dlon") / 2), 2)
    )
    .withColumn(
        "distancia_km",
        2 * f.lit(6371.0)
        * f.asin(
            f.sqrt(
                f.least(f.lit(1.0), f.col("a"))
            )
        )
    )
    .withColumn(
        "tiempo_horas",
        (
            f.col("BaseDateTime").cast("long")
            - f.col("tiempo_anterior").cast("long")
        ) / 3600.0
    )
    .withColumn(
        "velocidad_implicita_kmh",
        f.when(
            f.col("tiempo_horas") > 0,
            f.col("distancia_km") / f.col("tiempo_horas")
        )
    )
)

- Se cuantifica el problema de saltos entre posiciones

In [0]:
perfil_saltos = (
    distancias
    .agg(
        f.max("distancia_km").alias("salto_max_km"),
        f.expr(
            "percentile_approx(distancia_km, 0.99)"
        ).alias("p99_distancia_km"),
        f.max("velocidad_implicita_kmh")
            .alias("velocidad_implicita_max"),
        f.expr(
            "percentile_approx(velocidad_implicita_kmh, 0.99)"
        ).alias("p99_velocidad_implicita")
    )
)

display(perfil_saltos)

salto_max_km,p99_distancia_km,velocidad_implicita_max,p99_velocidad_implicita
8229.633864090267,0.894745724977534,1741592.5827008984,37.04974737851336


- El 99 % de las posiciones consecutivas están separadas por menos de 0,895 km. Sin embargo, existen saltos de hasta 8.229,6 km
- El p99 de velocidad implícita es de 37,05 km/h (≈20 nudos).
- Estos datos explican porque aparecen distancias de millones de kilómetros acumulados en una semana.
- Para hacer el cálculo de distancias por semana, se realizará un filtrado excluyendo los valores mayores a 37,05 km/h. Esto, con motivo de evitar la influencia de valores extremos.

In [0]:
P99_VELOCIDAD_KMH = 37.04974737851336

distancias_filtradas = (
    distancias
    .filter(
        (f.col("velocidad_implicita_kmh") <= P99_VELOCIDAD_KMH) &
        (f.col("velocidad_implicita_kmh").isNotNull())
    )
)

In [0]:
distancia_por_buque_filtrada = (
    distancias_filtradas
    .groupBy("MMSI")
    .agg(
        f.sum("distancia_km").alias("distancia_total_km")
    )
)

top10_distancia_filtrada = (
    distancia_por_buque_filtrada
    .orderBy(f.desc("distancia_total_km"))
    .limit(10)
)

display(
    top10_distancia_filtrada
    .withColumn(
        "distancia_total_km",
        f.round("distancia_total_km", 2)
    )
)

MMSI,distancia_total_km
367638030,5770.91
369355000,4389.55
367003380,4386.39
310531000,4088.81
311001223,4029.74
338760000,3825.49
367799380,3787.05
636020078,3786.28
205042000,3782.89
311073800,3777.96


- Después de filtrar los desplazamientos cuya velocidad implícita supera el percentil 99 (37,05 km/h), se obtienen distancias acumuladas considerablemente más razonables que en el cálculo inicial.
- El buque con MMSI 367638030 presenta la mayor distancia estimada durante la semana, con 5.770,91 km, seguido por 369355000 con 4.389,55 km y 367003380 con 4.386,39 km.

### d. ¿Dónde se concentra el tráfico?
Top 10 celdas de una grilla espacial (H3 resolución 8, o su propia rejilla lat/lon) por número de posiciones, y cuáles de
esas celdas corresponden a puertos del World Port Index.

- Para identificar las zonas con mayor concentración de tráfico AIS se utiliza una rejilla regular basada en latitud y longitud. Esta alternativa permite agrupar espacialmente las posiciones mediante funciones nativas de Spark, evitando dependencias adicionales.
- Cada posición se asigna a una celda y posteriormente se contabiliza el número de registros contenidos en cada una.

In [0]:
posiciones_grilla = (
    ais
    .select("LAT", "LON")
    .withColumn(
        "celda_lat",
        f.floor(f.col("LAT") * 10) / 10
    )
    .withColumn(
        "celda_lon",
        f.floor(f.col("LON") * 10) / 10
    )
)

In [0]:
trafico_grilla = (
    posiciones_grilla
    .groupBy("celda_lat", "celda_lon")
    .agg(
        f.count("*").alias("numero_posiciones")
    )
)

top10_celdas = (
    trafico_grilla
    .orderBy(f.desc("numero_posiciones"))
    .limit(10)
)

display(top10_celdas)

celda_lat,celda_lon,numero_posiciones
47.6,-122.4,1182304
29.7,-95.1,967489
33.7,-118.3,689167
40.6,-74.1,545776
26.0,-80.2,537013
29.7,-95.2,535797
32.7,-117.3,510478
29.9,-90.1,482917
26.1,-80.2,413780
25.7,-80.2,397522


- se encuentra una concentración grande en la celda (47.6, -122.4) debido a que supera 1,18 millones de posiciones, en segunda posición tenemos la celda (29.7, -95.1) con 967 mil posiciones.

### e. ¿Qué proporción de los buques de la semana transmitió los 7 días? ¿Dónde están los "visitantes de un solo día"?

- Para determinar la permanencia temporal de los buques durante la semana, se obtiene primero una única combinación por MMSI y fecha.
- Posteriormente, se cuenta el número de días distintos en los que cada buque transmitió información

Primero se reduce el dataset a combinaciones únicas MMSI-fecha

In [0]:
buque_dia = (
    ais
    .select(
        "MMSI",
        f.to_date("BaseDateTime").alias("fecha")
    )
    .dropDuplicates(["MMSI", "fecha"])
)

Se realiza el conteo de los días de presencia de cada buque

In [0]:
dias_por_buque = (
    buque_dia
    .groupBy("MMSI")
    .agg(
        f.count("*").alias("dias_transmitidos")
    )
)

Generación del reporte

In [0]:
resumen_permanencia = (
    dias_por_buque
    .agg(
        f.count("*").alias("buques_semana"),

        f.sum(
            f.when(f.col("dias_transmitidos") == 7, 1)
             .otherwise(0)
        ).alias("buques_7_dias"),

        f.sum(
            f.when(f.col("dias_transmitidos") == 1, 1)
             .otherwise(0)
        ).alias("buques_1_dia")
    )
    .withColumn(
        "proporcion_7_dias",
        f.round(
            f.col("buques_7_dias") /
            f.col("buques_semana") * 100,
            2
        )
    )
)

display(resumen_permanencia)

buques_semana,buques_7_dias,buques_1_dia,proporcion_7_dias
31871,12667,5996,39.74


- Se encuentra que  transitaron 31.871 buques distintos durante la semana.
- De esos buques distintos, 12.667 transmitieron durante los 7 días (39,74 % del total).
- Por otro lado, 5.996 buques transmitieron solamente durante un día (18,81 % del total).

Dónde estan los visitanes de un solo día

- Inicialmente, se hace el filtrado de los buques que transmitieron un solo día

In [0]:
visitantes_1_dia = (
    dias_por_buque
    .filter(f.col("dias_transmitidos") == 1)
    .select("MMSI")
)

- debido a que se tienen dos tablas de tamaños muy diferentes 60,5 millones de registros y 5.996 registros, se hace un broadcast para unir las coordenadas de dichos buques

In [0]:
posiciones_visitantes = (
    ais
    .select("MMSI", "LAT", "LON")
    .join(
        f.broadcast(visitantes_1_dia),
        on="MMSI",
        how="inner"
    )
)

Localización de los visitantes de un día

In [0]:
distribucion_visitantes = (
    posiciones_visitantes
    .withColumn(
        "celda_lat",
        f.floor(f.col("LAT") * 10) / 10
    )
    .withColumn(
        "celda_lon",
        f.floor(f.col("LON") * 10) / 10
    )
    .groupBy("celda_lat", "celda_lon")
    .agg(
        f.count("*").alias("numero_posiciones")
    )
)

Visualización de la distribución de visitantes de un solo día

In [0]:
display(distribucion_visitantes)

celda_lat,celda_lon,numero_posiciones
38.9,-62.8,1
37.7,-76.2,42
44.2,-68.3,48
41.4,-63.0,28
43.4,-70.5,35
39.8,-74.2,676
45.5,-84.1,8
41.8,-70.5,8
41.4,-62.8,3
41.4,-63.4,15


Databricks visualization. Run in Databricks to view.

- Se encuentra que hay barcos que reportaron ubicaciones por fuera del área marítima de estados unidos,para un análisis posterior se puede buscar información de dicha delimitación para hacer el filtrado de datos teniendo en cuenta el componente espacial.